# Typography and density

> Change how much fits without changing what the interface is.

Density is not a smaller version of everything. A dialog can keep readable text and fit more cards by tightening its gaps; a reading pane can increase its text size without inflating every control. Corners need not grow with either. AnswerStyle keeps three independent bases: `--text`, `--space`, and `--radius`. The same bases work for a page, an embedded tool, or one region.

In [ ]:
from answerstyle.preview import preview

## Three scales

The signed names tell you which side of the base you are on. At the default `--space: 1rem`, `--space-2` is .5rem and `--space2` is 2rem. Controls use those steps as their internal padding; a layout uses them as the gaps between controls. The same base has three quarters, one quarter, and larger steps above it. Text steps are .875, 1, 1.25, 1.5 and 2 times the base. Radius steps are one half, one and two times the base; the pill radius is independent.

Custom properties resolve on the element declaring them. The bases inherit; the derived steps are declared throughout the themed scope, so a region that sets its own base also gets the right derived values. Set the base to scale the entire region. Change one step on an element only when that one element needs an exception.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer answerstyle.tokens {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope):root {
      --font: system-ui, sans-serif;
      --font-mono: ui-monospace, Menlo, Consolas, monospace;
      --text: 1rem;
      --weight: 400; --weight1: 600;
      --leading: 1.5; --leading-1: 1.25;
      --prose-scale: .75;
      --prose-font-scale: 1;
      --prose-leading: 1.45;
      --prose-measure: 65ch;
      --space: 1rem;
      --radius: .5rem; --radius-full: 9999px;
      --relief: 1;
      --motion: 150ms;
      --ease: cubic-bezier(.2, 0, 0, 1);
    }

    :where(:scope, *), ::before, ::after {
      --text-1: calc(var(--text) * .875);
      --text1: calc(var(--text) * 1.25);
      --text2: calc(var(--text) * 1.5);
      --text3: calc(var(--text) * 2);
      --space-3: calc(var(--space) * .25);
      --space-2: calc(var(--space) * .5);
      --space-1: calc(var(--space) * .75);
      --space1: calc(var(--space) * 1.5);
      --space2: calc(var(--space) * 2);
      --space3: calc(var(--space) * 3);
      --radius-1: calc(var(--radius) * .5);
      --radius1: calc(var(--radius) * 2);
    }

    @media (prefers-reduced-motion: reduce) { :where(:scope):root { --motion: 0ms; } }
    @keyframes ui-spin { to { rotate: 1turn; } }
  }
}
'''

## The document is not a widget

Use ordinary headings, links, lists and text. Their margins are cleared so a stack or grid can decide their spacing. The document gets the system body font, heading sizes, and a readable measure where one is requested. An unmarked table or form control is still a native control, not a skinned component. A base reset must not remove a browser's native focus indicator.

A `[ui]` region can change the bases, colour scheme, or hue without adding a second stylesheet. Put `[unstyle]` around an opaque widget, including its popup host. It stops AnswerStyle's direct rules, not ordinary CSS inheritance. A `[ui]` region can restart styling further inside it.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer answerstyle.reset {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope, *), ::before, ::after { box-sizing: border-box; }
    :where(body, h1, h2, h3, h4, h5, h6, p, ul, ol, dl, dd, blockquote, pre, figure, figcaption, hr) { margin: 0; }
    :where(button, input, textarea, select) { font: inherit; }
    :where(img, picture, video, canvas, svg) { max-inline-size: 100%; block-size: auto; }
  }
}
@layer answerstyle.base {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope):is([ui], :root) { background: var(--color-background); color: var(--color-text); accent-color: var(--color-accent); }
    :where(:scope, *):is([ui], body) { font: var(--weight) var(--text)/var(--leading) var(--font); }
    :where(h1, h2, h3, h4, h5, h6) { font-family: var(--font-heading, inherit); font-weight: var(--weight1); line-height: var(--leading-1); text-wrap: balance; }
    :where(h1) { font-size: var(--text3); }
    :where(h2) { font-size: var(--text2); }
    :where(h3) { font-size: var(--text1); }
    :where(h4, h5, h6) { font-size: var(--text); }
    :where(a) { color: var(--color-accent); text-underline-offset: .15em; }
    :where(code, kbd, pre, samp) { font-family: var(--font-mono); }
    :where(small) { font-size: var(--text-1); }
    :where(svg.ui-icon, svg[icon], svg:has(> use)) { inline-size: 1.25em; block-size: 1.25em; flex: none; vertical-align: -.25em; }
    .ui-divider { border: 0; border-block-start: 1px solid var(--color-border); }
  }
}
'''

## Relief is independent of density

A raised action and a recessed input read differently before you click them. Both faces use the same imagined light from above and to the left. A raised rim is light above and dark below; the well flips it. The rim colours are relative to the actual face, not two fixed ladder steps: a darker rung in the light scheme becomes a lighter rung in the dark scheme.

Use `--relief` to scale those shadows and rims without changing labels or borders. At zero a control still has its own face. The drop shadow under a dialog or menu does not scale with relief; even a flat theme needs an overlay to look separate from the surface it covers.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer answerstyle.tokens {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope, *)[ui], :where(:scope):root { --face: var(--color-surface); }
    :where(:scope, *), ::before, ::after {
      --tint: oklch(from currentColor l c h / .08);
      --hi: oklch(from var(--face) calc(l + .16) c h / var(--relief));
      --lo: oklch(from var(--face) calc(l - .16) c h / var(--relief));
      --shadow-color: light-dark(rgb(0 0 0 / calc(.12 * var(--relief))), rgb(0 0 0 / calc(.5 * var(--relief))));
      --floating-shadow: light-dark(rgb(0 0 0 / .12), rgb(0 0 0 / .5));
      --bevel: inset 1px 1px 0 var(--hi), inset -1px -1px 0 var(--lo);
      --bevel-in: inset 1px 1px 0 var(--lo), inset -1px -1px 0 var(--hi);
      --raised: 0 2px 4px var(--shadow-color);
      --sunken: inset 0 2px 4px var(--shadow-color);
      --floating: 0 4px 12px var(--floating-shadow), 0 12px 32px var(--floating-shadow);
    }
  }
}
'''

## Density by composition

Both settings panels use the same markup. Only the region's bases change. Compare keeping the text size legible while varying the space and corner geometry with shrinking all three indiscriminately.

In [ ]:
def panel(density, sizes):
    return f'''
    <section ui style="{sizes}" card stack>
      <h3>{density}</h3>
      <p>Manage settings for this workspace.</p>
      <label stack style="--gap:var(--space-3)">Display name
        <input input value="Ada Lovelace">
      </label>
      <div cluster><button type="button" btn>Save</button><a href="#help">Help</a></div>
    </section>
    '''

spacious = panel('Spacious', '')
compact = panel('Compact', '--space:.75rem;--radius:.375rem')
preview(f'<div grid>{spacious}{compact}</div>', height='17rem')

## Prose has its own rhythm

Paragraphs, headings, tables and lists in an article are not an arrangement of controls. Put `[prose]` on a region so their spacing follows the written structure. A paragraph immediately followed by a list reads as an introduction; a heading starts a new section; a code block stays compact without turning the UI's overall leading down. The first and last blocks don't introduce extra space at the container edges.

Prose type derives from the region's `--text`; `--prose-font-scale` and `--prose-scale` tune the size and rhythm independently. The default line measure is `65ch`. A notebook can use the full available width and closer leading without changing the surrounding UI.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer answerstyle.base {
/* Typography folded in from typrose, based on Tailwind Labs' typography styles.
 * MIT License
 * Copyright (c) 2026 Answer.AI
 *
 * Permission is hereby granted, free of charge, to any person obtaining a copy
 * of this software and associated documentation files (the "Software"), to deal
 * in the Software without restriction, including without limitation the rights
 * to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
 * copies of the Software, and to permit persons to whom the Software is
 * furnished to do so, subject to the following conditions:
 *
 * The above copyright notice and this permission notice shall be included in all
 * copies or substantial portions of the Software.
 *
 * THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
 * IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
 * FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
 * AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
 * LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
 * OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
 * SOFTWARE.
 */

@scope (:root, [ui]) to ([unstyle]) {
  :where(:scope, *)[prose] {
    --sp: calc(var(--prose-scale) * 1em);
    max-inline-size: var(--prose-measure); overflow-wrap: anywhere;
    font-size: calc(var(--text) * var(--prose-font-scale)); line-height: var(--prose-leading); color: var(--color-text);

    /* Top margins establish the rhythm; the container owns outer spacing. */
    :where(p, ol, ul, dl) { margin: var(--sp) 0 0; }
    :where(dt) { margin-top: var(--sp); font-weight: 600; }
    :where(dd) { margin-top: calc(var(--sp) * .25); padding-inline-start: 1.375em; }
    :where(li) { margin: calc(var(--sp) * .25) 0 0; }
    :where(ol > li, ul > li) { padding-inline-start: .375em; }
    :where(ol, ul) { padding-inline-start: 1.7em; }
    :where(ol) { list-style-type: decimal; }
    :where(ul) { list-style-type: disc; }
    :where(ol[type="A"], ol[type="A" s]) { list-style-type: upper-alpha; }
    :where(ol[type="a"], ol[type="a" s]) { list-style-type: lower-alpha; }
    :where(ol[type="I"], ol[type="I" s]) { list-style-type: upper-roman; }
    :where(ol[type="i"], ol[type="i" s]) { list-style-type: lower-roman; }
    :where(ol[type="1"]) { list-style-type: decimal; }
    :where(ul ul, ul ol, ol ul, ol ol) { margin: 0; }
    > :where(ul, ol) > li :where(p) { margin: calc(var(--sp) * .5) 0 0; }
    > :where(ul, ol) > li > :where(p:first-child) { margin-top: calc(var(--sp) * .75); }
    :where(ol > li)::marker { font-weight: 400; color: var(--color-muted); }
    :where(ul > li)::marker { color: var(--color-muted); }
    :where(p + ul, p + ol) { margin-top: calc(var(--sp) * .25); }

    :where(h1) { font-size: 2.25em; font-weight: 800; line-height: 1.1; margin: calc(var(--sp) * 1.4) 0 0; }
    :where(h2) { font-size: 1.5em; font-weight: 700; line-height: 1.3; margin: calc(var(--sp) * 1.4) 0 0; }
    :where(h3) { font-size: 1.25em; font-weight: 600; line-height: 1.4; margin: calc(var(--sp) * 1.2) 0 0; }
    :where(h4, h5, h6) { font-size: 1em; font-weight: 600; line-height: 1.4; margin: calc(var(--sp) * 1.25) 0 0; }
    :where(h1 strong) { font-weight: 900; color: inherit; }
    :where(h2 strong) { font-weight: 800; color: inherit; }
    :where(h3 strong, h4 strong, h5 strong, h6 strong) { font-weight: 700; color: inherit; }
    :where(.lead) { font-size: 1.25em; line-height: 1.6; color: var(--color-muted); margin: var(--sp) 0 0; }

    :where(a:not([btn])) { color: var(--color-accent); text-decoration: underline; text-underline-offset: .15em; font-weight: 500; }
    :where(a:not([btn]):focus-visible) { outline: var(--ring); outline-offset: 2px; }
    :where(a strong, a code) { color: inherit; }
    :where(strong) { font-weight: 600; }

    :where(img, picture, video, figure) { margin: calc(var(--sp) * 1.5) 0 0; }
    :where(picture) { display: block; }
    :where(img, video) { display: block; max-inline-size: 100%; block-size: auto; border-radius: var(--radius); }
    :where(figcaption) { font-size: .875em; line-height: 1.4; color: var(--color-muted); margin-top: calc(var(--sp) * .5); }
    :where(small) { font-size: .875em; color: var(--color-muted); }

    :where(blockquote) {
      font-style: italic; font-weight: 500; margin: calc(var(--sp) * 1.5) 0 0; padding-inline-start: 1em;
      border-inline-start: .25rem solid var(--color-border); quotes: "\201C" "\201D" "\2018" "\2019";
    }
    :where(blockquote strong, blockquote code) { color: inherit; }

    :where(code, pre, samp) { font-family: var(--font-mono); }
    :where(*:not(pre) > code) {
      font-size: .875em; font-weight: 600; background: var(--color-surface);
      padding: .1em .3rem; border-radius: .25rem; line-height: 1.2rem;
    }
    :where(h1 code, h2 code, h3 code, h4 code, h5 code, h6 code, blockquote code, thead th code) { color: inherit; }
    :where(pre) {
      font-size: inherit; line-height: 1.25; white-space: pre-wrap; overflow-wrap: anywhere; overflow-x: auto;
      padding: .25rem .3rem; border-radius: 5px; background: var(--color-surface); margin: var(--sp) 0 0;
    }
    :where(pre code) { padding: 0; background: none; font: inherit; }
    :where(kbd) {
      font-size: .875em; font-weight: 500; font-family: inherit; padding: .125em .3125em; border-radius: .3125rem;
      box-shadow: 0 0 0 1px var(--color-border), 0 3px 0 var(--color-border);
    }
    :where(mark) { background: var(--v-3); color: var(--v2); padding: .1em .2em; border-radius: var(--radius-1); }

    /* Prose tables wrap and scroll, independently of the application [table] skin. */
    :where(table) {
      display: block; width: fit-content; max-width: calc(100% + 1em); overflow-x: auto; border-collapse: collapse;
      font-size: .875em; line-height: 1.5; margin: calc(var(--sp) * 2) -.5em 0;
    }
    :where(caption) { width: max-content; max-width: 100%; padding: 0 .5em; }
    :where(th, td) { text-align: start; overflow-wrap: break-word; }
    :where(thead) { border-bottom: 1px solid var(--color-border); }
    :where(thead th) { font-weight: 600; vertical-align: bottom; padding: 0 .5em calc(var(--sp) * .5); }
    :where(thead th strong, thead th code) { color: inherit; }
    :where(tbody tr) { border-bottom: 1px solid var(--color-border); }
    :where(tbody tr:last-child) { border-bottom: 0; }
    :where(tbody td) { vertical-align: baseline; padding: calc(var(--sp) * .5) .5em; }
    :where(tfoot) { border-top: 1px solid var(--color-border); }
    :where(tfoot td) { vertical-align: top; padding: calc(var(--sp) * .5) .5em; }

    :where(hr) {
      border: 0; margin-top: var(--sp); position: relative; height: 1.25rem;
      background: radial-gradient(circle, transparent 6px, var(--color-border) 6px 8px, transparent 8px) center / 20px 20px no-repeat;
    }
    :where(hr)::before, :where(hr)::after {
      content: ''; position: absolute; top: 50%; width: calc(50% - .75rem); border-bottom: 1px solid var(--color-border);
    }
    :where(hr)::before { inset-inline-start: 0; }
    :where(hr)::after { inset-inline-end: 0; }

    :where(h1 + *) { margin-top: calc(var(--sp) * 1.5); }
    :where(h2 + *) { margin-top: var(--sp); }
    :where(h3 + *, h4 + *, h5 + *, h6 + *) { margin-top: calc(var(--sp) * .5); }
    :where(picture > img, figure > :not(figcaption)) { margin-top: 0; margin-bottom: 0; }
    > :first-child { margin-top: 0; }
    > :last-child { margin-bottom: 0; }
  }

  [prose-size=sm] { --prose-scale: .7; --prose-font-scale: .875; }
  [prose-size=lg] { --prose-scale: .8; --prose-font-scale: 1.125; }
  [prose-size=xl] { --prose-scale: .9; --prose-font-scale: 1.25; }
  [prose-size="2xl"] { --prose-scale: 1.1; --prose-font-scale: 1.5; }
}
}
'''

In [ ]:
article = '''<h2>Escaping a crowded pane</h2>
<p>Give a passage room to read, not every control room to sprawl. In this pane, we can:</p>
<ul><li>Keep prose at a useful measure.</li><li>Use the full width for tables when needed.</li></ul>
<pre><code>parts = walk(workspace)
print(len(parts))</code></pre>
<p>The next section starts without a trailing margin on the article.</p>'''
preview(f'''<div grid style="--grid-min:17rem">
  <article card prose style="--prose-measure:28ch">{article}</article>
  <article card prose style="--text:.875rem;--prose-leading:1.3;--prose-measure:none">{article}</article>
</div>''', height='23rem')

## Fonts belong to the theme

The default font stack uses system faces. For the IBM Plex house style, load `fonts.css` and set `--font: 'IBM Plex Sans', system-ui, sans-serif` and `--font-mono: 'IBM Plex Mono', ui-monospace, monospace`. Set `--font-heading` for a region that wants Plex Serif headings. Fonts are optional assets, not dependencies of layout or scale.

In [ ]:
#| hide
#| asset fonts.css
_fonts = r'''
@font-face { font-family: 'IBM Plex Sans'; font-weight: 400; font-style: normal; font-display: swap; src: url('./fonts/ibm-plex-sans-latin-400-normal.woff2') format('woff2'); }
@font-face { font-family: 'IBM Plex Sans'; font-weight: 400; font-style: italic; font-display: swap; src: url('./fonts/ibm-plex-sans-latin-400-italic.woff2') format('woff2'); }
@font-face { font-family: 'IBM Plex Sans'; font-weight: 600; font-style: normal; font-display: swap; src: url('./fonts/ibm-plex-sans-latin-600-normal.woff2') format('woff2'); }
@font-face { font-family: 'IBM Plex Serif'; font-weight: 600; font-style: normal; font-display: swap; src: url('./fonts/ibm-plex-serif-latin-600-normal.woff2') format('woff2'); }
@font-face { font-family: 'IBM Plex Mono'; font-weight: 400; font-style: normal; font-display: swap; src: url('./fonts/ibm-plex-mono-latin-400-normal.woff2') format('woff2'); }
'''
